In [0]:
# ============================================================

# GOLD – dim_source

# Grain: source

# ============================================================

from pyspark.sql import functions as F

from pyspark.sql.window import Window

# ---------------- CONFIG ----------------

SILVER_TICKETS = "workspace.slainte_silver.easyvista_tickets_silver_demo"

GOLD_DB = "workspace.slainte_gold"

DIM_SOURCE = f"{GOLD_DB}.dim_source"

# ---------------- SETUP ----------------

spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

df_tickets = spark.table(SILVER_TICKETS)

# ---------------- BUILD DIM ----------------

df_source_base = (

    df_tickets

    .select(F.col("source").alias("source_name"))

    .filter(F.col("source_name").isNotNull())

    .distinct()

)

window = Window.orderBy("source_name")

df_dim_source = (

    df_source_base

    .withColumn("source_id", F.row_number().over(window))

    .withColumn("created_at", F.current_timestamp())

    .select(

        "source_id",

        "source_name"

    )

)

# ---------------- WRITE ----------------

df_dim_source.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(DIM_SOURCE)

print("✅ dim_source created successfully")

display(spark.table(DIM_SOURCE).orderBy("source_id"))
 